# Brands

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DoubleType, DateType, TimestampType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

#Define schema for the data files
brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), False),
    StructField("category_code", StringType(), False),
])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw_data/brands/*.csv"

df = spark.read.option('header', "true").option("delimiter", ",").schema(brand_schema).csv(raw_data_path)

# add metadata columns

df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
        .withColumn("injected_at", F.current_timestamp())

display(df.limit(5))

brand_code,brand_name,category_code,_source_file,injected_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw_data/brands/brands.csv,2026-04-01T22:41:21.982Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw_data/brands/brands.csv,2026-04-01T22:41:21.982Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw_data/brands/brands.csv,2026-04-01T22:41:21.982Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw_data/brands/brands.csv,2026-04-01T22:41:21.982Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw_data/brands/brands.csv,2026-04-01T22:41:21.982Z


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

# Category

In [0]:
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), False),
])
raw_data_path = "/Volumes/ecommerce/source_data/raw_data/category/*.csv"

df = spark.read.option('header', "true").option("delimiter", ",").schema(category_schema).csv(raw_data_path)

# add metadata columns

df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
        .withColumn("injected_at", F.current_timestamp())

display(df.limit(5))


category_code,category_name,_source_file,injected_at
ce,Electronics,dbfs:/Volumes/ecommerce/source_data/raw_data/category/category.csv,2026-04-01T22:50:44.261Z
app,Apparel,dbfs:/Volumes/ecommerce/source_data/raw_data/category/category.csv,2026-04-01T22:50:44.261Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw_data/category/category.csv,2026-04-01T22:50:44.261Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw_data/category/category.csv,2026-04-01T22:50:44.261Z
bks,Books,dbfs:/Volumes/ecommerce/source_data/raw_data/category/category.csv,2026-04-01T22:50:44.261Z


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_categories")

# Products

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw_data/products/*.csv"

df = spark.read.option('header', "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path)

# add metadata columns

df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
        .withColumn("injected_at", F.current_timestamp())

display(df.limit(5))
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_products")


product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp,_source_file,injected_at
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:51.617Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:51.617Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:51.617Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:51.617Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:51.617Z


# Customers

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

raw_data_path = "/Volumes/ecommerce/source_data/raw_data/customers/*.csv"

df = spark.read.option('header', "true").option("delimiter", ",").schema(customers_schema).csv(raw_data_path)

# add metadata columns

df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
        .withColumn("injected_at", F.current_timestamp())

display(df.limit(5))
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")


customer_id,phone,country_code,country,state,_source_file,injected_at
CUST000000000001,917280033536.0,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw_data/customers/customers.csv,2026-04-01T23:07:30.959Z
CUST000000000002,619489725433.0,AU,Australia,VIC,dbfs:/Volumes/ecommerce/source_data/raw_data/customers/customers.csv,2026-04-01T23:07:30.959Z
CUST000000000003,919390066524.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw_data/customers/customers.csv,2026-04-01T23:07:30.959Z
CUST000000000004,917073741793.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw_data/customers/customers.csv,2026-04-01T23:07:30.959Z
CUST000000000005,618478772532.0,AU,Australia,WA,dbfs:/Volumes/ecommerce/source_data/raw_data/customers/customers.csv,2026-04-01T23:07:30.959Z


Date

In [0]:
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

raw_data_path = "/Volumes/ecommerce/source_data/raw_data/date/*.csv"

df = spark.read.option('header', "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# add metadata columns

df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
        .withColumn("injected_at", F.current_timestamp())

display(df.limit(5))
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_date")


date,year,day_name,quarter,week_of_year,_source_file,injected_at
01-08-2025,2025,friday,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:32.489Z
02-08-2025,2025,SATURDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:32.489Z
03-08-2025,2025,SUNDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:32.489Z
04-08-2025,2025,MONDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:32.489Z
05-08-2025,2025,TUESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:32.489Z
